# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from nba_api.stats.static import players

from src.config import *
from src.utils import *
from src.feature_builder import *
from src.feature_aggregation import *

#display full columns
pd.set_option('display.max_column', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_seq_items', None)
# pd.set_option('display.max_colwidth', 500)
# pd.set_option('expand_frame_repr', True)

In [2]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-11-01 14:09:27.803456


# 🔁 Chargement des fichiers

In [3]:
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)
df_boxscores = pd.read_csv(boxscores_file, dtype={'gameId': str})
df_games = pd.read_csv(games_file, dtype={'GAME_ID': str, 'TEAM_ID': str})


In [4]:
games_file

'data/raw_last/games_merged/games_merged_all_seasons_2025-11-01_03-03-49.csv'

In [5]:
df_boxscores

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage
0,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1897,Metta,World Peace,M. World Peace,metta-world-peace,F,NaN,NaN,29:20,83.3,112.3,-28.9,-28.9,0.077,1.00,6.7,0.038,0.031,0.034,6.7,0.583,0.582,0.203,95.73,95.73,60.0,0.156,0.467,0.267,0.249,0.475,0.186,0.100,0.370,2,2,4,10,17,14,13,36,2,0,0.917,0.083,0.933,0.267,0.000,0.267,0.067,0.133,0.667,0.429,0.571,0.0,0.0,0.429,0.571,7,12,0,1,0.000,1,2,0.50,1,1,2,1,3,2,1,2,15,-14.0,0.350,0.267,0.00,0.200,0.125,0.167,0.125,0.059,0.080,0.125,0.071,0.75,0.667,0.400,0.143,0.000,0.300
1,0020000279,1610612741,Chicago,Bulls,CHI,bulls,2033,Marcus,Fizer,M. Fizer,marcus-fizer,F,NaN,NaN,37:09,90.3,101.4,-11.1,-11.1,0.087,0.50,11.8,0.111,0.128,0.120,23.5,0.300,0.368,0.181,92.38,92.38,72.0,0.043,0.475,0.217,0.223,0.477,0.297,0.156,0.361,2,4,2,6,19,14,10,42,3,0,1.000,0.000,0.750,0.000,0.000,0.250,0.250,0.250,0.750,0.333,0.667,0.0,0.0,0.333,0.667,3,10,0,0,0.000,2,2,1.00,4,5,9,2,0,0,4,2,8,-7.0,0.115,0.167,0.00,0.000,0.250,0.154,0.400,0.217,0.273,0.167,0.250,0.00,0.000,0.600,0.095,0.000,0.123
2,0020000279,1610612741,Chicago,Bulls,CHI,bulls,1434,Dragan,Tarlac,D. Tarlac,dragan-tarlac,C,NaN,NaN,22:49,85.1,122.4,-37.3,-37.3,0.000,0.00,0.0,0.150,0.154,0.152,75.0,1.000,1.064,0.151,100.98,100.98,47.0,-0.007,0.486,0.286,0.259,0.500,0.260,0.095,0.364,0,4,0,2,16,10,19,34,0,0,1.000,0.000,0.500,0.000,0.000,0.000,0.500,0.000,0.500,0.000,1.000,0.0,0.0,0.000,1.000,1,1,0,0,0.000,2,2,1.

In [6]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22010,1610612738.0,BOS,Boston Celtics,0021000001,2010-10-26,BOS vs. MIA,W,239,88,32,69,0.464,8,16,0.500,16,25,0.640,8,34,42,25,6,4,18,19,8.0,2010-11
1,22010,1610612756.0,PHX,Phoenix Suns,0021000002,2010-10-26,PHX @ POR,L,239,92,36,74,0.486,9,19,0.474,11,16,0.688,7,23,30,15,3,4,19,19,-14.0,2010-11
2,22010,1610612745.0,HOU,Houston Rockets,0021000003,2010-10-26,HOU @ LAL,L,240,110,38,91,0.418,8,20,0.400,26,28,0.929,16,37,53,25,6,7,20,25,-2.0,2010-11
3,22010,1610612757.0,POR,Portland Trail Blazers,0021000002,2010-10-26,POR vs. PHX,W,240,106,43,93,0.462,10,20,0.500,10,15,0.667,18,30,48,31,11,2,12,22,14.0,2010-11
4,22010,1610612748.0,MIA,Miami Heat,0021000001,2010-10-26,MIA @ BOS,L,242,80,27,74,0.365,8,20,0.400,18,25,0.720,11,28,39,15,10,6,17,21,-8.0,2010-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66921,22025,1610612737.0,ATL,Atlanta Hawks,0022500020,2025-10-31,ATL @ IND,W,239,128,51,94,0.543,10,30,0.333,16,20,0.800,12,43,55,30,10,5,10,23,20.0,2025-26
66922,22025,1610612763.0,MEM,Memphis Grizzlies,0022500024,2025-10-31,MEM vs. LAL,NaN,0,8,3,3,1.000,1,1,1.000,1,2,0.500,0,2,2,3,1,0,1,0,4.0,2025-26
66923,22025,1610612747.0,LAL,Los Angeles Lakers,0022500024,2025-10-31,LAL @ MEM,NaN,0,4,2,4,0.500,0,2,0.000,0,0,NaN,0,0,0,1,1,0,2,2,-4.0,2025-26
66924,22025,1610612739.0,CLE,Cleveland Cavaliers,0022500022,2025-10-31,CLE vs. TOR,NaN,241,93,33,80,0.413,13,42,0.310,14,22,0.636,12,25,37,22,9,10,12,14,-6.4,2025-26


# 🧼 Nettoyage des minutes jouées

In [7]:
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    try:
        parts = str(val).split(':')
        return int(parts[0]) + int(parts[1]) / 60 if len(parts) == 2 else float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['minutes'].apply(convert_minutes)

# Rename gameId and teamId in boxscores

In [8]:
rename_columns = {
    'gameId': 'GAME_ID',
    'teamId': 'TEAM_ID',
}

df_boxscores.rename(columns=rename_columns, inplace=True)


In [9]:
# Normaliser GAME_ID et TEAM_ID pour garantir des jointures cohérentes
for df in (df_games, df_boxscores):
    for col in ['GAME_ID', 'TEAM_ID']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        df[col] = df[col].astype(str).replace('<NA>', None)

In [10]:
print(df_games.dtypes)


SEASON_ID              int64
TEAM_ID               object
TEAM_ABBREVIATION     object
TEAM_NAME             object
GAME_ID               object
GAME_DATE             object
MATCHUP               object
WL                    object
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
SEASON                object
dtype: object


In [11]:
print(df_boxscores.dtypes)

GAME_ID                                object
TEAM_ID                                object
teamCity                               object
teamName                               object
teamTricode                            object
                                       ...   
percentageBlocksAllowed_usage         float64
percentagePersonalFouls_usage         float64
percentagePersonalFoulsDrawn_usage    float64
percentagePoints_usage                float64
MINUTES_PLAYED                        float64
Length: 101, dtype: object


# Merge GAME_DATE dans les boxscores et remove les duplicates créés

In [12]:
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
df_boxscores = df_boxscores.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

df_boxscores = df_boxscores.drop_duplicates(subset=['GAME_ID', 'TEAM_ID', 'playerSlug'])

In [13]:
# display nat GAME_DATE in df_boxscores
# check for NaT values in GAME_DATE
nat_dates = df_boxscores[df_boxscores['GAME_DATE'].isna()]
nat_dates

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED,GAME_DATE


In [14]:
missing = df_boxscores[df_boxscores['GAME_DATE'].isna()][['GAME_ID', 'TEAM_ID']].drop_duplicates()

print(f"Exemples de lignes avec GAME_DATE NaT:")
print(missing.head(10))

# On regarde s’ils existent dans df_games
merged_check = missing.merge(df_games[['GAME_ID', 'TEAM_ID']], on=['GAME_ID', 'TEAM_ID'], how='left', indicator=True)
print(merged_check['_merge'].value_counts())


Exemples de lignes avec GAME_DATE NaT:
Empty DataFrame
Columns: [GAME_ID, TEAM_ID]
Index: []
_merge
left_only     0
right_only    0
both          0
Name: count, dtype: int64


In [15]:
print("Exemples d’ID manquants dans df_games")
print(missing[~missing.set_index(['GAME_ID', 'TEAM_ID']).index.isin(df_games.set_index(['GAME_ID', 'TEAM_ID']).index)])


Exemples d’ID manquants dans df_games
Empty DataFrame
Columns: [GAME_ID, TEAM_ID]
Index: []


# 🔄 Cast dynamique des colonnes numériques


In [16]:
numeric_cols = df_boxscores.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)

In [17]:
df_boxscores

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED,GAME_DATE
0,20000279,1610612741,Chicago,Bulls,CHI,bulls,1897,Metta,World Peace,M. World Peace,metta-world-peace,F,NaN,0.0,29:20,83.3,112.3,-28.9,-28.9,0.077,1.00,6.7,0.038,0.031,0.034,6.7,0.583,0.582,0.203,95.73,95.73,60.0,0.156,0.467,0.267,0.249,0.475,0.186,0.100,0.370,2,2,4,10,17,14,13,36,2,0,0.917,0.083,0.933,0.267,0.000,0.267,0.067,0.133,0.667,0.429,0.571,0.0,0.0,0.429,0.571,7,12,0,1,0.000,1,2,0.50,1,1,2,1,3,2,1,2,15,-14.0,0.350,0.267,0.00,0.200,0.125,0.167,0.125,0.059,0.080,0.125,0.071,0.75,0.667,0.400,0.143,0.000,0.300,29.333333,2000-12-08
1,20000279,1610612741,Chicago,Bulls,CHI,bulls,2033,Marcus,Fizer,M. Fizer,marcus-fizer,F,NaN,0.0,37:09,90.3,101.4,-11.1,-11.1,0.087,0.50,11.8,0.111,0.128,0.120,23.5,0.300,0.368,0.181,92.38,92.38,72.0,0.043,0.475,0.217,0.223,0.477,0.297,0.156,0.361,2,4,2,6,19,14,10,42,3,0,1.000,0.000,0.750,0.000,0.000,0.250,0.250,0.250,0.750,0.333,0.667,0.0,0.0,0.333,0.667,3,10,0,0,0.000,2,2,1.00,4,5,9,2,0,0,4,2,8,-7.0,0.115,0.167,0.00,0.000,0.250,0.154,0.400,0.217,0.273,0.167,0.250,0.00,0.000,0.600,0.095,0.000,0.123,37.150000,2000-12-08
2,20000279,1610612741,Chicago,Bulls,CHI,bulls,1434,Dragan,Tarlac,D. Tarlac,dragan-tarlac,C,NaN,0.0,22:49,85.1,122.4,-37.3,-37.3,0.000,0.00,0.0,0.150,0.154,0.152,75.0,1.000,1.064,0.151,100.98,100.98,47.0,-0.007,0.486,0.286,0.259,0.500,0.260,0.095,0.364,0,4,0,2,16,10,19,34,0,0,1.000,0.000,0.500,0.000,0.000,0.000,0.500,0

# Build player features (players and top players absence for each game)

In [18]:
player_features_df = build_player_status_features(df_boxscores)

today = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


os.makedirs(DATA_PLAYERS_DIR, exist_ok=True)
player_features_output = os.path.join(DATA_PLAYERS_DIR, f'player_features_{today}.csv')

# to csv
#player_features_df.to_csv(player_features_output, index=False)

In [19]:
player_features_df

,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
0,20000279,1610612741,2000-12-08,1897,1,0,0,0,0,0,0,22.9,5.5,18.5,0.156,
1,20000279,1610612741,2000-12-08,2033,1,0,0,0,0,0,0,17.8,-2.0,8.6,0.043,
2,20000279,1610612741,2000-12-08,1434,1,0,0,0,0,0,0,7.4,-3.5,-1.2,-0.007,
3,20000279,1610612741,2000-12-08,2064,1,0,0,0,0,0,0,28.8,-0.5,26.6,0.188,
4,20000279,1610612741,2000-12-08,1500,1,0,0,0,0,0,0,20.3,0.5,17.7,0.037,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784245,22500092,1610612761,2025-10-24,1642347,1,0,0,0,0,0,0,11.0,1.5,10.8,0.099,
784246,22500092,1610612761,2025-10-24,1642367,1,0,0,0,0,0,0,2.4,1.5,-1.0,0.071,
784247,22500092,1610612761,2025-10-24,1642419,1,0,0,0,0,0,0,11.4,0.5,9.8,0.182,
784248,22500092,1610612761,2025-10-24,202066,0,1,0,0,0,0,1,0.0,0.0,0.0,0.000,dnp - coach's decision


In [20]:
#print duplicates on 'GAME_ID' and 'TEAM_ID', personId
duplicates = player_features_df[player_features_df.duplicated(subset=['GAME_ID', 'TEAM_ID', 'personId'], keep=False)]
if not duplicates.empty:
    print("Duplicates found:")
    display(duplicates)

In [21]:
#player_features_df[player_features_df['is_absent'] == 1].sort_values(by='GAME_ID').head(50)

#same but filter with comment "DNP - Coach's Decision"

filtered = player_features_df[
    (player_features_df['is_absent'] == 1) &
    (~player_features_df['comment'].str.contains("coach's decision", na=False))
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(10))

filtered = player_features_df[
    (player_features_df['is_personal'] == 1) &
    (~player_features_df['comment'].str.contains("personal", na=False)) &
    (~player_features_df['comment'].str.contains("not with team", na=False)) 
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(10))




# display(player_features_df[
#     player_features_df['is_personal'] == 1 &
#     (~player_features_df['comment'].str.contains("nwt", na=False))
#     ].head(50))


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
714168,52300201,1610612741,2024-04-19,1641763,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
714169,52300201,1610612741,2024-04-19,1630172,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
714182,52300201,1610612748,2024-04-19,202710,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
714183,52300201,1610612748,2024-04-19,1626196,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
714184,52300201,1610612748,2024-04-19,1626179,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
748643,52400101,1610612737,2025-04-15,1630552,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnp - injury/illness
748697,52400111,1610612748,2025-04-16,1631107,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
748699,52400111,1610612748,2025-04-16,201567,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd-return to competition reconditioning
748764,52400201,1610612737,2025-04-18,1630552,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - injury/illness
748752,52400201,1610612748,2025-04-18,201567,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - personal


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
354289,21200144,1610612738,2012-11-18,2545,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
355943,21200607,1610612745,2013-01-21,202962,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
381385,21201169,1610612756,2013-04-10,201563,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
365720,21201182,1610612739,2013-04-12,203079,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matters
442619,21400441,1610612757,2014-12-26,2549,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family
432403,21400871,1610612737,2015-02-28,201143,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family reasons
81018,40100154,1610612759,2002-05-01,1495,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - death in family
133330,40400311,1610612759,2005-05-22,299,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family business
133344,40400312,1610612759,2005-05-24,299,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family business
133381,40400313,1610612759,2005-05-28,299,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family business


In [22]:

# Analyse des statuts des joueurs
total_rows = len(player_features_df)

# Comptage des cas
n_present = player_features_df['is_present'].sum()
n_absent = player_features_df['is_absent'].sum()
n_injured = player_features_df['is_injured'].sum()

# is_resting
# is_suspended
# is_personal

n_resting = player_features_df['is_resting'].sum()
n_suspended = player_features_df['is_suspended'].sum()
n_personal = player_features_df['is_personal'].sum()

# Lignes incohérentes : aucune des colonnes n'est True

mask_no_status = (
    (player_features_df['is_present'] == 0) &
    (player_features_df['is_absent'] == 0) &
    (player_features_df['is_injured'] == 0) &
    (player_features_df['is_resting'] == 0) &
    (player_features_df['is_suspended'] == 0) &
    (player_features_df['is_personal'] == 0)
)


n_inconsistent = mask_no_status.sum()

# Lignes avec plusieurs statuts à la fois (logiquement impossible)
mask_multiple_status = (
    player_features_df[['is_present', 'is_absent', 'is_injured']].sum(axis=1) > 1
)
n_multiple = mask_multiple_status.sum()

print(f"✅ Analyse des statuts des joueurs sur {total_rows} lignes")
print(f" - Joueurs présents : {n_present}")
print(f" - Joueurs absents : {n_absent}")
print(f" - Joueurs blessés : {n_injured}")
print(f" - Joueurs en repos : {n_resting}")
print(f" - Joueurs suspendus : {n_suspended}")
print(f" - Joueurs pour raisons personnelles : {n_personal}")
print(f"❌ Lignes sans statut défini : {n_inconsistent}")
print(f"⚠️ Lignes avec plusieurs statuts actifs : {n_multiple}")

display(player_features_df[mask_no_status].head(10))



✅ Analyse des statuts des joueurs sur 754146 lignes
 - Joueurs présents : 618911
 - Joueurs absents : 135235
 - Joueurs blessés : 21327
 - Joueurs en repos : 392
 - Joueurs suspendus : 1037
 - Joueurs pour raisons personnelles : 1005
❌ Lignes sans statut défini : 0
⚠️ Lignes avec plusieurs statuts actifs : 21327


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment


# ⚙️ Aggrégation des data Player par équipe et match


In [23]:
# df_team_players = aggregate_team_player_features(player_features_df, top_n=10)

# display(df_team_players.head(10))
# display(df_team_players.tail(10))

# #display some lines with absent players
# absent_players = df_team_players[df_team_players['num_injured'] >= 1]
# absent_players 

# Identification des top joueurs sur l'ensemble du dataset
df_top_players = identify_historical_top_players(player_features_df)
print(f"✅ {df_top_players['is_historical_top'].sum()} top joueurs identifiés sur {len(df_top_players)} joueurs.")


df_agg_actual = aggregate_actual_team_features(player_features_df)
print(f"✅ {df_agg_actual.shape[0]} lignes générées dans l'aggregation actuelle par équipe.")

df_agg_top_abs = flag_top_players_absences(player_features_df, df_top_players)
print(f"✅ {df_agg_top_abs['top_player_absent'].sum()} absences de top joueurs détectées.")

df_team_features_final = df_agg_actual.merge(df_agg_top_abs, on=["GAME_ID", "TEAM_ID"], how="left")
df_team_features_final.fillna(0, inplace=True)

#add flags has_top_absent flag to use it as input and not roll it to see in analyse how much it helps to scrap this data before match
df_team_features_final['has_top_absent'] = df_team_features_final['top_player_absent'].apply(lambda x: 1 if x > 0 else 0)
df_team_features_final['has_absent'] = df_team_features_final['num_absent'].apply(lambda x: 1 if x > 0 else 0)


#sort by GAME_ID, TEAM_ID and GAME_DATE
df_team_features_final.sort_values(by=['GAME_ID', 'TEAM_ID', 'GAME_DATE'], inplace=True)


print(f"✅ Fusion réussie. Shape finale : {df_team_features_final.shape}")


/home/ju/Documents/Dev/NBA_Predictor/src/feature_aggregation.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tops = roll.groupby(['GAME_ID', 'TEAM_ID']).apply(pick_top).reset_index()


✅ 308062 top joueurs identifiés sur 308062 joueurs.
✅ 63875 lignes générées dans l'aggregation actuelle par équipe.
✅ 19309 absences de top joueurs détectées.
✅ Fusion réussie. Shape finale : (63875, 27)


In [24]:
# display(df_team_features_final.head(10))
# display(df_team_features_final.tail(10))

# top_player_absence_rate
# top_player_injury_rate
# top_player_resting_rate
# top_player_suspension_rate
# top_player_personal_rate

# display(df_team_features_final[df_team_features_final['top_player_absent_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_injury_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_resting_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_suspension_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_personal_rate'] > 0])
display(df_team_features_final)


,GAME_ID,TEAM_ID,GAME_DATE,player_perf_score_mean,player_perf_score_sum,num_present,num_absent,num_injured,num_suspended,num_resting,num_personal,num_absent_other,top_player_count,top_player_absent,top_player_injured,top_player_resting,top_player_suspended,top_player_personal,top_player_absent_other,top_player_absent_rate,top_player_injury_rate,top_player_resting_rate,top_player_suspension_rate,top_player_personal_rate,top_player_absent_other_rate,has_top_absent,has_absent
0,20000001,1610612752,2000-10-31,10.450000,125.4,12,0,0,0,0,0,0,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1,20000001,1610612755,2000-10-31,15.658333,187.9,12,0,0,0,0,0,0,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2,20000002,1610612739,2000-10-31,13.866667,166.4,11,1,0,0,0,0,1,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,20000002,1610612751,2000-10-31,14.950000,179.4,10,2,0,0,0,0,2,5,1,0,0,0,0,1,0.2,0.0,0.0,0.0,0.0,0.2,1,1
4,20000003,1610612753,2000-10-31,14.616667,175.4,10,2,0,0,0,0,2,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63870,52400131,1610612758,2025-04-16,14.253846,185.3,12,1,0,0,0,0,1,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
63871,52400201,1610612737,2025-04-18,18.150000,217.8,9,3,1,0,0,0,2,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
63872,52400201,1610612748,2025-04-18,15.686667,235.3,9,6,0,0,0,1,5,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
63873,52400211,1610612742,2025-04-18,14.438462,187.7,11,2,0,0,0,0,2,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


# ⚙️ Aggrégation des data Boxscore purs par équipe et match


## 📊 Agrégation des données par équipe et match


In [25]:

# 1. Agrégation par somme
group_keys = ['GAME_ID', 'TEAM_ID']
sum_agg = df_boxscores[group_keys + cols_to_sum].copy()
sum_agg = sum_agg.groupby(group_keys).sum().reset_index()

# 2. Agrégation pondérée par les minutes jouées
weighted_agg = compute_weighted_mean_features(df_boxscores, group_keys, cols_to_weighted_avg, weight_col='MINUTES_PLAYED')

# 3. Fusion des deux agrégats
team_match_stats = pd.merge(sum_agg, weighted_agg, on=group_keys, how='left')

# 4. Identifier l'équipe adverse
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
def get_opponent(row):
    teams = teams_in_game.get(row['GAME_ID'], [])
    opps = [tid for tid in teams if tid != row['TEAM_ID']]
    return opps[0] if opps else None

team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(get_opponent, axis=1)


In [26]:
count = 0

for game_id, teams in teams_in_game.items():
    if len(teams) < 2:
        #print(f"Attention : GAME_ID {game_id} a moins de 2 équipes : {teams}")
        count += 1
        
print(f"Nombre de GAME_ID avec moins de 2 équipes : {count}")



Nombre de GAME_ID avec moins de 2 équipes : 393


# 🔁 Ajout des colonnes OPP_ avec les data de l'adversaire

In [27]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]
df_opp = team_match_stats.rename(columns={col: f"OPP_{col}" for col in team_cols}).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'})
match_dataset = pd.merge(
    team_match_stats,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [28]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'fieldGoalsMade_traditional',
       'fieldGoalsAttempted_traditional', 'threePointersMade_traditional',
       'threePointersAttempted_traditional', 'freeThrowsMade_traditional',
       'freeThrowsAttempted_traditional', 'reboundsOffensive_traditional',
       'reboundsDefensive_traditional',
       ...
       'OPP_percentageReboundsTotal_usage', 'OPP_percentageAssists_usage',
       'OPP_percentageTurnovers_usage', 'OPP_percentageSteals_usage',
       'OPP_percentageBlocks_usage', 'OPP_percentageBlocksAllowed_usage',
       'OPP_percentagePersonalFouls_usage',
       'OPP_percentagePersonalFoulsDrawn_usage', 'OPP_percentagePoints_usage',
       'OPP_plusMinusPoints_traditional'],
      dtype='object', length=165)

# 🏠 Ajout IS_HOME et IS_WIN


In [29]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22010,1610612738,BOS,Boston Celtics,21000001,2010-10-26,BOS vs. MIA,W,239,88,32,69,0.464,8,16,0.500,16,25,0.640,8,34,42,25,6,4,18,19,8.0,2010-11
1,22010,1610612756,PHX,Phoenix Suns,21000002,2010-10-26,PHX @ POR,L,239,92,36,74,0.486,9,19,0.474,11,16,0.688,7,23,30,15,3,4,19,19,-14.0,2010-11
2,22010,1610612745,HOU,Houston Rockets,21000003,2010-10-26,HOU @ LAL,L,240,110,38,91,0.418,8,20,0.400,26,28,0.929,16,37,53,25,6,7,20,25,-2.0,2010-11
3,22010,1610612757,POR,Portland Trail Blazers,21000002,2010-10-26,POR vs. PHX,W,240,106,43,93,0.462,10,20,0.500,10,15,0.667,18,30,48,31,11,2,12,22,14.0,2010-11
4,22010,1610612748,MIA,Miami Heat,21000001,2010-10-26,MIA @ BOS,L,242,80,27,74,0.365,8,20,0.400,18,25,0.720,11,28,39,15,10,6,17,21,-8.0,2010-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66921,22025,1610612737,ATL,Atlanta Hawks,22500020,2025-10-31,ATL @ IND,W,239,128,51,94,0.543,10,30,0.333,16,20,0.800,12,43,55,30,10,5,10,23,20.0,2025-26
66922,22025,1610612763,MEM,Memphis Grizzlies,22500024,2025-10-31,MEM vs. LAL,NaN,0,8,3,3,1.000,1,1,1.000,1,2,0.500,0,2,2,3,1,0,1,0,4.0,2025-26
66923,22025,1610612747,LAL,Los Angeles Lakers,22500024,2025-10-31,LAL @ MEM,NaN,0,4,2,4,0.500,0,2,0.000,0,0,NaN,0,0,0,1,1,0,2,2,-4.0,2025-26
66924,22025,1610612739,CLE,Cleveland Cavaliers,22500022,2025-10-31,CLE vs. TOR,NaN,241,93,33,80,0.413,13,42,0.310,14,22,0.636,12,25,37,22,9,10,12,14,-6.4,2025-26


In [30]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [31]:
# Assurer le format datetime pour GAME_DATE
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])



# Merge avec les infos de match (MATCHUP, SEASON)
match_dataset = match_dataset.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP', 'SEASON','GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

In [32]:

print("match_dataset after merge")
display(match_dataset)

match_dataset after merge


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [33]:

#display lines with nan values on matchup
print("Lines with NaN in MATCHUP:")
display(match_dataset[match_dataset['MATCHUP'].isna()])

Lines with NaN in MATCHUP:


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [34]:
# Définir si l'équipe joue à domicile
match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs', na=False).astype(int)

# Calcul du résultat (win) et écart de points
match_dataset['IS_WIN'] = (match_dataset['points_traditional'] > match_dataset['OPP_points_traditional']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['points_traditional'] - match_dataset['OPP_points_traditional']

# 

In [35]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

# SPECIAL FOR ODDS : Keep only season past 2010-11


In [36]:
# # get first line with season 2010-11 and cut all lines before
# first_season = match_dataset[match_dataset['SEASON'] == '2010-11'].index[0]
# match_dataset = match_dataset.iloc[first_season:].reset_index(drop=True)
# match_dataset

# Merge Odds with dataset

In [37]:
all_odds_df = merge_odds_csv_files(DATA_ODDS_HISTORY_DIR)

match_dataset = match_odds_with_dataset_test(all_odds_df, match_dataset)



------------------ Nombre de lignes sans cotes (home/away): 25772 ------------------


In [38]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

# Merge match_dataset with df_team_features_final on TEAM and GAME

In [39]:

#team id as str
match_dataset['TEAM_ID'] = match_dataset['TEAM_ID'].astype(str)
match_dataset['GAME_ID'] = match_dataset['GAME_ID'].astype(str)

#date as datetime
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])

df_team_features_final['TEAM_ID'] = df_team_features_final['TEAM_ID'].astype(str)
df_team_features_final['GAME_ID'] = df_team_features_final['GAME_ID'].astype(str)
df_team_features_final['GAME_DATE'] = pd.to_datetime(df_team_features_final['GAME_DATE'])



# Merge match_dataset with df_team_features_final on TEAM and GAME, keep GAME_DATE
match_dataset = match_dataset.merge(
    df_team_features_final[['GAME_ID', 'TEAM_ID', 'GAME_DATE'] + df_team_features_final.columns.difference(['GAME_ID', 'TEAM_ID', 'GAME_DATE']).tolist()],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)



# match_dataset_ = match_dataset.merge(
#     df_team_features_final,
#     on=['GAME_ID', 'TEAM_ID'],
#     how='left'
# )
#remove duplicates
match_dataset = match_dataset.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])



In [40]:
#sort by GAME_DATE and GAME_ID
#match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)



#print GAME_DATE_x and GAME_DATE_y to check if they are the same
print("GAME_DATE_x and GAME_DATE_y are the same:", (match_dataset['GAME_DATE_x'] == match_dataset['GAME_DATE_y']).all())

#drop GAME_DATE_x and GAME_DATE_y to GAME_DATE
match_dataset['GAME_DATE'] = match_dataset['GAME_DATE_x']
match_dataset = match_dataset.drop(columns=['GAME_DATE_x', 'GAME_DATE_y'])



#display cols in common match_dataset and df_team_features_final
common_cols = set(match_dataset.columns).intersection(set(df_team_features_final.columns))
print("Common columns between match_dataset and df_team_features_final:")
display(common_cols)


GAME_DATE_x and GAME_DATE_y are the same: True
Common columns between match_dataset and df_team_features_final:


{'GAME_DATE',
 'GAME_ID',
 'TEAM_ID',
 'has_absent',
 'has_top_absent',
 'num_absent',
 'num_absent_other',
 'num_injured',
 'num_personal',
 'num_present',
 'num_resting',
 'num_suspended',
 'player_perf_score_mean',
 'player_perf_score_sum',
 'top_player_absent',
 'top_player_absent_other',
 'top_player_absent_other_rate',
 'top_player_absent_rate',
 'top_player_count',
 'top_player_injured',
 'top_player_injury_rate',
 'top_player_personal',
 'top_player_personal_rate',
 'top_player_resting',
 'top_player_resting_rate',
 'top_player_suspended',
 'top_player_suspension_rate'}

# Reorder columns for visualisation

In [41]:
cols_first = ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN','POINT_DIFF','has_absent','has_top_absent','num_absent','top_player_absent', 'points_traditional','ODDS','OPP_ODDS']
other_cols = [col for col in match_dataset.columns if col not in cols_first]
match_dataset = match_dataset[cols_first + other_cols]
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,has_absent,has_top_absent,num_absent,top_player_absent,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt

# Add OPP cols for player stats

In [42]:
opp_cols = [f"OPP_{col}" for col in cols_player_stats]
df_opp = match_dataset.rename(columns={col: f"OPP_{col}" for col in cols_player_stats})

match_dataset = pd.merge(
    match_dataset,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [43]:
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,has_absent,has_top_absent,num_absent,top_player_absent,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt

# 🚀 Features avancées


In [44]:

# Ajouter les features avancées aux stats de match
#match_dataset = add_advanced_boxscore_features(match_dataset)

# MOVED TO CONFIG
# features_to_roll = cols_to_sum + cols_to_weighted_avg
# features_to_roll += [f"OPP_{col}" for col in cols_to_sum + cols_to_weighted_avg]

# Calcul des features glissantes shiftées
match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], features_to_roll, N_LIST, method="ewm")

match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], top_player_features_to_roll, N_LIST_TOP, method="ewm", apply_log=True)


match_dataset = compute_winrates(match_dataset, "TEAM_ID", "IS_WIN", "IS_HOME", N_LIST)
match_dataset = compute_win_ratio(match_dataset, "TEAM_ID", "IS_WIN", N_LIST)

match_dataset["IS_WIN_SHIFTED"] = match_dataset.groupby("TEAM_ID")["IS_WIN"].shift(1).fillna(0).astype(int)
match_dataset["WIN_STREAK"] = match_dataset.groupby("TEAM_ID").apply(
    lambda x: compute_win_streak(x, "TEAM_ID", "IS_WIN_SHIFTED")).reset_index(level=0, drop=True)
match_dataset = compute_side_win_streak(match_dataset, win_shifted_col="IS_WIN_SHIFTED")


# Calcul des jours de repos pour l'équipe et l'adversaire
match_dataset["DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "TEAM_ID")
match_dataset["OPP_DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "OPP_TEAM_ID")

# Avantage de repos
match_dataset["REST_ADVANTAGE"] = (
    match_dataset["DAYS_SINCE_LAST_GAME"] - match_dataset["OPP_DAYS_SINCE_LAST_GAME"]
)


match_dataset = compute_rolling_rest_advantage(
    match_dataset, "TEAM_ID", "IS_HOME", "REST_ADVANTAGE", N_LIST
)

match_dataset = compute_home_away_pts(
    match_dataset, "TEAM_ID", "IS_HOME", "points_traditional", "OPP_points_traditional", N_LIST
)

# match_dataset = rename_pts_against_columns(match_dataset)

match_dataset = compute_h2h(match_dataset, N_LIST)
# match_dataset = compute_h2h_pts_margin(match_dataset, N_LIST)
match_dataset = compute_h2h_season(match_dataset)
match_dataset = compute_h2h_streak(match_dataset)

match_dataset = compute_elo(match_dataset)
match_dataset = compute_elo_season(match_dataset)

match_dataset = convert_elos_to_elo_diff(match_dataset)


/home/ju/Documents/Dev/NBA_Predictor/src/feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[roll_col] = (
/home/ju/Documents/Dev/NBA_Predictor/src/feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[roll_col] = (
/home/ju/Documents/Dev/NBA_Predictor/src/feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.

In [45]:
pd.options.display.max_columns = None
display(match_dataset)

GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      20000001 2000-10-31  1610612752  1610612755        1       0   
1      20000001 2000-10-31  1610612755  1610612752        0       1   
2      20000002 2000-10-31  1610612739  1610612751        0       1   
3      20000002 2000-10-31  1610612751  1610612739        1       0   
4      20000003 2000-10-31  1610612753  1610612764        1       1   
...         ...        ...         ...         ...      ...     ...   
63870  22500022 2025-10-31  1610612761  1610612739        0       1   
63871  22500023 2025-10-31  1610612741  1610612752        1       0   
63872  22500023 2025-10-31  1610612752  1610612741        0       0   
63873  22500024 2025-10-31  1610612747  1610612763        0       0   
63874  22500024 2025-10-31  1610612763  1610612747        1       0   

       POINT_DIFF  has_absent  has_top_absent  num_absent  top_player_absent  \
0           -29.0           0               0           0                  0   
1            29.0           0               0           0                  0   
2             4.0           1               0           1                  0   
3            -4.0           1               1           2                  1   
4            11.0           1               0           2                  0   
...           ...         ...             ...         ...                ...   
63870        11.0           1               1           2                  1   
63871         0.0           1               1          13                  5   
63872         0.0           1               1          14                  5   
63873         0.0           1               1          11                  5   
63874         0.0           1               1          14                  5   

       points_traditional  ODDS  OPP_ODDS  fieldGoalsMade_traditional  \
0                      72   NaN       NaN                          25   
1                     101   NaN       NaN                          38   
2                      86   NaN       NaN                          32   
3                      82   NaN       NaN                          31   
4                      97   NaN       NaN                          34   
...                   ...   ...       ...                         ...   
63870                 112   NaN       NaN                          40   
63871                   0   NaN       NaN                           0   
63872                   0   NaN       NaN                           0   
63873                   0   NaN       NaN                           0   
63874                   0   NaN       NaN                           0   

       fieldGoalsAttempted_traditional  threePointersMade_traditional  \
0                                   70                              3   
1                                   66                              3   
2                                   78                              2   
3                                   85                              3   
4                                   79                              6   
...                                ...                            ...   
63870                               82                             13   
63871                                0                              0   
63872                                0                              0   
63873                                0                              0   
63874                                0                              0   

       threePointersAttempted_traditional  freeThrowsMade_traditional  \
0                                      11                          19   
1                                       8                          22   
2                                       7                          20   
3                                      10                          17   
4                                      16                          23   
..

# 🧽 Nettoyage et sauvegarde


In [46]:

#TODO : see why not found in config.py after reboot
player_absent_input_cols = [
    'has_top_absent',
    'has_absent',
    'top_player_absent',
    'num_absent',
]

final_date = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
raw_save_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')
clean_save_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

os.makedirs(DATA_FINAL_DATASET_DIR, exist_ok=True)
os.makedirs(DATA_FINAL_CLEANED_DATASET_DIR, exist_ok=True)

match_dataset.to_csv(raw_save_path, index=False)

cols_to_drop = features_to_roll+top_player_features_to_roll+COLS_MATCH_REAL
#remove num_absent and top_player_absent from cols_to_drop to see if we can use them has input
cols_to_drop = [col for col in cols_to_drop if col not in player_absent_input_cols]


final_cleaned = match_dataset.drop(columns=cols_to_drop, errors='ignore')
final_cleaned.to_csv(clean_save_path, index=False)

print(f"✅ Fichier brut : {raw_save_path}")
print(f"✅ Fichier clean : {clean_save_path}")

final_cleaned

✅ Fichier brut : data/final_dataset/nba_features_final_2025-11-01_14-16-47.csv
✅ Fichier clean : data/final_cleaned_dataset/nba_features_cleaned_final_2025-11-01_14-16-47.csv


GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      20000001 2000-10-31  1610612752  1610612755        1       0   
1      20000001 2000-10-31  1610612755  1610612752        0       1   
2      20000002 2000-10-31  1610612739  1610612751        0       1   
3      20000002 2000-10-31  1610612751  1610612739        1       0   
4      20000003 2000-10-31  1610612753  1610612764        1       1   
...         ...        ...         ...         ...      ...     ...   
63870  22500022 2025-10-31  1610612761  1610612739        0       1   
63871  22500023 2025-10-31  1610612741  1610612752        1       0   
63872  22500023 2025-10-31  1610612752  1610612741        0       0   
63873  22500024 2025-10-31  1610612747  1610612763        0       0   
63874  22500024 2025-10-31  1610612763  1610612747        1       0   

       POINT_DIFF  has_absent  has_top_absent  num_absent  top_player_absent  \
0           -29.0           0               0           0                  0   
1            29.0           0               0           0                  0   
2             4.0           1               0           1                  0   
3            -4.0           1               1           2                  1   
4            11.0           1               0           2                  0   
...           ...         ...             ...         ...                ...   
63870        11.0           1               1           2                  1   
63871         0.0           1               1          13                  5   
63872         0.0           1               1          14                  5   
63873         0.0           1               1          11                  5   
63874         0.0           1               1          14                  5   

       ODDS  OPP_ODDS   SEASON  ROLL_fieldGoalsMade_traditional_3  \
0       NaN       NaN  2000-01                                NaN   
1       NaN       NaN  2000-01                                NaN   
2       NaN       NaN  2000-01                                NaN   
3       NaN       NaN  2000-01                                NaN   
4       NaN       NaN  2000-01                                NaN   
...     ...       ...      ...                                ...   
63870   NaN       NaN  2025-26                          42.582343   
63871   NaN       NaN  2025-26                          46.261745   
63872   NaN       NaN  2025-26                          37.998877   
63873   NaN       NaN  2025-26                          42.578127   
63874   NaN       NaN  2025-26                          42.319896   

       ROLL_fieldGoalsMade_traditional_5  ROLL_fieldGoalsMade_traditional_10  \
0                                    NaN                                 NaN   
1                                    NaN                                 NaN   
2                                    NaN                                 NaN   
3                                    NaN                                 NaN   
4                                    NaN                                 NaN   
...                                  ...                                 ...   
63870                          43.365185                           43.887769   
63871                          44.727231                           43.815233   
63872                          37.998301                           38.392047   
63873                          41.527665                           40.310418   
63874                          41.558244                           41.064642   

       ROLL_fieldGoalsMade_traditional_25  ROLL_fieldGoalsMade_traditional_50  \
0                                     NaN                                 NaN   
1                                     NaN                                 NaN   
2                                     NaN                                 NaN   
3                                     NaN                                 NaN   

In [47]:
final_cleaned.columns

Index(['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN',
       'POINT_DIFF', 'has_absent', 'has_top_absent', 'num_absent',
       ...
       'H2H_LAST_100_COUNT', 'H2H_LAST_200_DIFF', 'H2H_LAST_200_WINRATE',
       'H2H_LAST_200_COUNT', 'H2H_SEASON_WINS', 'H2H_SEASON_MATCHES',
       'H2H_SEASON_WINRATE', 'H2H_WIN_STREAK', 'ELO_DIFF', 'ELO_DIFF_SEASON'],
      dtype='object', length=1464)

In [48]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-11-01 14:20:01.630715
Total time:  0:10:33.827259
